In [1]:
import sys
sys.path.append('/Users/rifatordulu/Developer/lstm-stock-price-prediction')
# sys.path.append('/Users/rifatordulu/Developer/lstm-stock-price-prediction/helpers')

%run "../helpers/data_manipulator.py"
%run "../helpers/yfinance_data_fetcher.py"

from custom_objects import register_custom_objects
# Register globally
register_custom_objects()

### LOAD VECTORS FOR TODAY

In [2]:
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import MinMaxScaler
from keras.models import load_model

SEQUENCE_SIZE = 65
# stocks_to_check = ["AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "TSLA"]
# stocks_to_check = my_stocks

combos = [["AAPL", "Next-Day-Close-To-Next-Day-Open-Ratio", 3], # Very good
          ["AAPL", "Next-Day-High-To-Next-Day-Open-Ratio", 1], # Slightly good
          ["AMZN", "Next-Day-High-To-Next-Day-Open-Ratio", 1], # Slightly good
          ["BABA", "Next-Day-Close-To-Next-Day-Open-Ratio", 1], # Slightly good
          ["CRM", "Next-Day-Close-To-Next-Day-Open-Ratio", 3], # Very good
          ["CRM", "Next-Day-High-To-Next-Day-Open-Ratio", 3], # Very good
          ["CSCO", "Next-Day-Close-To-Next-Day-Open-Ratio", 1], # slightly good
          ["CSCO", "Next-Day-High-To-Next-Day-Open-Ratio", 3], # Very good
          ["DASH", "Next-Day-High-To-Next-Day-Open-Ratio", 3], # Very good
          ["DIA", "Next-Day-High-To-Next-Day-Open-Ratio", 2], # Moderately good
          ["DIS", "Next-Day-Close-To-Next-Day-Open-Ratio", 3], # Very good
          ["DIS", "Next-Day-High-To-Next-Day-Open-Ratio", 3], # Very good
          ["F", "Next-Day-Close-To-Next-Day-Open-Ratio", 1], # Slightly good
          ["GOOG", "Next-Day-Close-To-Next-Day-Open-Ratio", 2], # Moderately good
          ["GOOG", "Next-Day-High-To-Next-Day-Open-Ratio", 2], # Moderetaly good
          ["INTC", "Next-Day-High-To-Next-Day-Open-Ratio", 1], # Slightly good
          ["IONQ", "Next-Day-Close-To-Next-Day-Open-Ratio", 1], # Slightly good
          ["JNJ", "Next-Day-Close-To-Next-Day-Open-Ratio", 1], # Slightly good
          ["JPM", "Next-Day-High-To-Next-Day-Open-Ratio", 2], # Moderately good
          ["KO", "Next-Day-Close-To-Next-Day-Open-Ratio", 3], # Very good
          ["KO", "Next-Day-High-To-Next-Day-Open-Ratio", 3], # Very good
          ["LLY", "Next-Day-High-To-Next-Day-Open-Ratio", 1], # Slightly good
          ["MSFT", "Next-Day-High-To-Next-Day-Open-Ratio", 1], # Slightly good
          ["MSTR", "Next-Day-Close-To-Next-Day-Open-Ratio", 3], # Very good
          ["NFLX", "Next-Day-High-To-Next-Day-Open-Ratio", 1], # Slightly good
          ["NVDA", "Next-Day-Close-To-Next-Day-Open-Ratio", 3], # Very good
          ["PFE", "Next-Day-Close-To-Next-Day-Open-Ratio", 3], # Very good
          ["PFE", "Next-Day-High-To-Next-Day-Open-Ratio", 3], # Very good
          ["PLTR", "Next-Day-Close-To-Next-Day-Open-Ratio", 3], # Very good
          ["PLTR", "Next-Day-High-To-Next-Day-Open-Ratio", 3], # Very good
          ["QQQ", "Next-Day-High-To-Next-Day-Open-Ratio", 1], # Slightlky good
          ["RGTI", "Next-Day-Close-To-Next-Day-Open-Ratio", 1], # Slightly good
          ["RGTI", "Next-Day-High-To-Next-Day-Open-Ratio", 1], # Slightly good
          ["SPOT", "Next-Day-Close-To-Next-Day-Open-Ratio", 2], # Moderately good
          ["SPY", "Next-Day-High-To-Next-Day-Open-Ratio", 1], # Slightly good
          ["UBER", "Next-Day-Close-To-Next-Day-Open-Ratio", 2], # Moderately good
          ["UBER", "Next-Day-High-To-Next-Day-Open-Ratio", 2], # Moderately good
          ["UNH", "Next-Day-High-To-Next-Day-Open-Ratio", 2], # Moderately good
          ["V", "Next-Day-Close-To-Next-Day-Open-Ratio", 1], # Slightly good
          ["WMT", "Next-Day-Close-To-Next-Day-Open-Ratio", 2], # Moderately good
          ["XOM", "Next-Day-Close-To-Next-Day-Open-Ratio", 1], # Slightly good
          ["ABNB", "Next-Day-Close-To-Next-Day-Open-Ratio", 3], # Very good
          ["ABNB", "Next-Day-High-To-Next-Day-Open-Ratio", 1], # Slightly good
          ["GLD", "Next-Day-High-To-Next-Day-Open-Ratio", 1], #
          ["TLT", "Next-Day-Close-To-Next-Day-Open-Ratio", 2], # Moderately good
          ["IWM", "Next-Day-Close-To-Next-Day-Open-Ratio", 1], # Slightly good
          ["IWM", "Next-Day-High-To-Next-Day-Open-Ratio", 3], # Very good
          ["", ""], #
                   ]

write_header = True
for combo in combos:
    ss = combo[0]
    tt = combo[1]
    print(f"SS: {ss}, TT: {tt}")
    score = combo[2]
    # Prepare model location and name
    model_location = "../models-more-CNN-last3mo-val/" + ss + "/"
    model_name = tt + ".keras"
    threshold = pd.read_csv(model_location + tt + "_threshold.csv").iloc[0]["Threshold"]
    target_val = pd.read_csv(model_location + tt + "_binary_target_val.csv").iloc[0]["Target-Val"]

    print(f"stock: {ss}, target: {tt}, threshold: {threshold}, target_val: {target_val}")
    
    # Load the best performing model
    best_model = load_model(model_location + model_name)
    
    #data_frame = construct_values_for_model(ticker_symbols = stocks_to_check, sequence_size=20, use_for_last_day_prediction=True, verbose=False, refresh=True)
    data_frame = construct_values_for_model(ticker_symbols = [ss], 
                                            underlying_target=tt,
                                            sequence_size=SEQUENCE_SIZE, 
                                            use_for_last_day_prediction=False, 
                                            verbose=False, 
                                            data_interval="2y",
                                            refresh=True)
    data_frame = get_specific_date_data(data_frame, "2025-01-16")
    # data_frame = get_specific_date_internal_data(data_frame, "2025-01-01", "2025-01-15")

    i = True
    if i:
        print(data_frame)
        print(data_frame["LstmData"])
        if pd.isna(data_frame.iloc[-1]['Next-Day-Open']):
                print(f"ABORTING.... NEXT DAY'S DATA IS NOT AVAILABLE FOR THE LAST DAY IN YOUR SELECTION...")
                break
        i = False
    
    # print(data_frame)
    X_test = np.array(data_frame["LstmData"].to_list())
    X_test_ticker = np.array(data_frame["Ticker"].to_list())
    X_test_sector = np.array(data_frame["Sector"].to_list())
    X_test_yesterday = data_frame[lstm_features].to_numpy()
    X_test_days = np.array(data_frame["Date"].to_list())
    y_test = np.array(data_frame["y-value"].to_list())
    
    # print(X_test[0])

    # print(data_frame.iloc[0]["LstmData"].loc[:,"Date"])
    # for i in range(len(data_frame)):
    #         print(f"Stock: {X_test_ticker[i]}, Date: {data_frame['Date'].iloc[i]}, y-value: {data_frame['y-value'].iloc[i]}")

    y_predict = best_model.predict([X_test, X_test_ticker, X_test_sector, X_test_yesterday])
    data_frame['y-predict'] = y_predict
    for ticker in data_frame["Orig_Ticker"].unique():
        data_frame.loc[data_frame["Orig_Ticker"] == ticker, 'y-predict-original'] = inverse_normalize_data(data_frame[['y-predict']], dm_target, ticker)

    data_frame["Target-Val"] = target_val
    data_frame["Threshold"] = threshold
    data_frame["Target"] = tt
    data_frame["Score"] = score
    data_frame["Final-Price"] = data_frame["Next-Day-Open"] * (1 + data_frame["Target-Val"])
    
    data_frame['y-predict-binary'] = data_frame['y-predict-original'] >= threshold
    print(f"PRINTING POSITIVE PREDICTIONS FOR: {ss} WITH TARGET: {tt}")
    print(data_frame[data_frame['y-predict-binary'] == 1.0])
    to_write = data_frame[["Orig_Ticker", "Date", "y-value", "y-predict", "y-predict-original", "y-predict-binary", "Score", "Threshold", "Target", "Target-Val", "Next-Day-Open", "Final-Price"]]
    to_write.to_csv("../todays-guess.csv", mode='a', index=False, header=write_header)
    to_write_only_true = to_write[to_write["y-predict-binary"] == 1.0]
    to_write_only_true.to_csv("../todays-guess_only_true.csv", mode='a', index=False, header=write_header)
    write_header = False

SS: AAPL, TT: Next-Day-Close-To-Next-Day-Open-Ratio
stock: AAPL, target: Next-Day-Close-To-Next-Day-Open-Ratio, threshold: 0.501278817653656, target_val: 0.0136174322432206


2025-01-18 15:59:45.037548: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2 Pro
2025-01-18 15:59:45.037568: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2025-01-18 15:59:45.037573: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2025-01-18 15:59:45.037588: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-01-18 15:59:45.037598: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


fetch_finance_data_for_tickers: about to load from yfinance: AAPL for interval: 2y


/Users/rifatordulu/Developer/lstm-stock-price-prediction/helpers/data_manipulator.py:133: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  data["Date"] = pd.to_datetime(data["Date"], errors="raise")
/Users/rifatordulu/python_3_11/lib/python3.11/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/Users/rifatordulu/python_3_11/lib/python3.11/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/Users/rifatordulu/python_3_11/lib/python3.11/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMax

object
{'AAPL': 0}
{'Tech': 0}
                                              LstmData  \
236       High-Low-Ratio  High-Open-Ratio  Close-Op...   

                          Date  Ticker  Sector  Target-20D-Mean  \
236  2025-01-16 00:00:00-05:00       0       0         0.001128   

     Target-20D-Sigma  y-value  High-Low-Ratio  High-Open-Ratio  \
236          0.012306      0.0        0.718831         0.513903   

     Close-Open-Ratio  ...  Next-Day-Open-To-Close      CMO3      CCI3  \
236           0.30851  ...                0.584553  0.205925  0.034535   

         CCI2      ROC4  AroonOsc3  Next-Day-Open  y-value-original  \
236  0.166667  0.481866   0.333333     232.119995               0.0   

     Orig_Ticker  Orig_Sector  
236         AAPL         Tech  

[1 rows x 35 columns]
236         High-Low-Ratio  High-Open-Ratio  Close-Op...
Name: LstmData, dtype: object


...

/Users/rifatordulu/Developer/lstm-stock-price-prediction/helpers/data_manipulator.py:133: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  data["Date"] = pd.to_datetime(data["Date"], errors="raise")
/Users/rifatordulu/python_3_11/lib/python3.11/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/Users/rifatordulu/python_3_11/lib/python3.11/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(
/Users/rifatordulu/python_3_11/lib/python3.11/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMax

{'IWM': 0}
{'Unknown': 0}
                                              LstmData  \
236       High-Low-Ratio  High-Open-Ratio  Close-Op...   

                          Date  Ticker  Sector  Target-20D-Mean  \
236  2025-01-16 00:00:00-05:00       0       0         0.007222   

     Target-20D-Sigma  y-value  High-Low-Ratio  High-Open-Ratio  \
236          0.006384      0.0        0.556576         0.524091   

     Close-Open-Ratio  ...  Next-Day-Open-To-Close      CMO3     CCI3  \
236          0.507584  ...                0.554345  0.782499  0.74536   

         CCI2      ROC4  AroonOsc3  Next-Day-Open  y-value-original  \
236  0.166667  0.517664   0.833333     226.929993               0.0   

     Orig_Ticker  Orig_Sector  
236          IWM      Unknown  

[1 rows x 35 columns]
236         High-Low-Ratio  High-Open-Ratio  Close-Op...
Name: LstmData, dtype: object


/Users/rifatordulu/Developer/lstm-stock-price-prediction/helpers/data_manipulator.py:491: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data_frame["Ticker"] = data_frame["Ticker"].replace(stock_dict)
/Users/rifatordulu/Developer/lstm-stock-price-prediction/helpers/data_manipulator.py:506: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data_frame["Sector"] = data_frame["Sector"].replace(sector_dict)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step
PRINTING POSITIVE PREDICTIONS FOR: IWM WITH TARGET: Next-Day-High-To-Next-Day-Open-Ratio
Empty DataFrame
Columns: [LstmData, Date, Ticker, Sector, Target-20D-Mean, Target-20D-Sigma, y-value, High-Low-Ratio, High-Open-Ratio, Close-Open-Ratio, Close-Prev-Close, MACD-Prev-MACD, 200D-Ratio, MACD_SIG, MFI3, RSI, RSI3, POCR, fastk, fastd, SlowK1, willr3, willr2, willr1, Next-Day-Open-To-Open, Next-Day-Open-To-Close, CMO3, CCI3, CCI2, ROC4, AroonOsc3, Next-Day-Open, y-value-original, Orig_Ticker, Orig_Sector, y-predict, y-predict-original, Target-Val, Threshold, Target, Score, Final-Price, y-predict-binary]
Index: []

[0 rows x 43 columns]
SS: , TT: 


IndexError: list index out of range

In [ ]:
print(data_frame.iloc[0]["LstmData"])

In [ ]:
x = [11,2,3,4,5,6,7,8,9,10]

df = pd.DataFrame({'x': x})

val = df['x'].rolling(window=5, min_periods=1).max().shift(-5)
print(val)

In [ ]:
# data_frame["y-predict"] = y_predict

# sector_threshold_mapping =  pd.read_csv(model_location + model_name + ".csv")
# print(sector_threshold_mapping)
# for sector in data_frame["Orig_Sector"].unique():
#     threshold = sector_threshold_mapping[sector_threshold_mapping["Sector"] == sector].iloc[0]["Threshold"]
#     data_frame.loc[data_frame["Orig_Sector"] == sector, "y-predict-binary"] = (data_frame[data_frame["Orig_Sector"] == sector]["y-predict"] > threshold).astype(float)

# print(f"\n\nResults: \n {data_frame.loc[(data_frame['y-predict-binary'] == 1) & ((data_frame['Target-20D-Mean'] + data_frame['Target-20D-Sigma']) > 0.011)]}")


In [ ]:
import numpy as np
X = np.load('X_values.npy')
y = np.load('y_values.npy')
dates = np.load('dates_values.npy', allow_pickle=True)
tickers = np.load('tickers_values.npy', allow_pickle=True)

In [ ]:
import pandas as pd


index = 1100
print(sum(y))

with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(X.shape)
    print(dates.shape)
    print(tickers.shape)
    arr = X[index,0:64,3]
    print(dates[index])
    print(tickers[index])
    print(y[index])
    for i in arr:
        print(i)
    # print(arr.min())